In [ ]:
from chardet import detect
import pandas as pd

def get_encoding_type(file):
    with open(file, 'rb') as f:
        rawdata = f.read()
    return detect(rawdata)['encoding']


region_file = './maakunnat_pinta_alat.csv'
accident_file ='./koeti_riista_tau_001_fi.csv'

region = pd.read_csv(region_file, encoding=get_encoding_type(region_file))
accident = pd.read_csv(accident_file, encoding=get_encoding_type(accident_file))

In [4]:
print('Region dataset')
print(region.head())
print('Game dataset')
print(accident.head())

Region dataset
   id      gml_id  natcode               namefin              nameswe  \
0   1  1601110001        1               Uusimaa               Nyland   
1   2  1601110027       15             Pohjanmaa          Österbotten   
2   3  1601110035       19                 Lappi             Lappland   
3   4  1601110011        7           Päijät-Häme  Päijänne-Tavastland   
4   5  1601110037       21  Ahvenanmaan maakunta     Landskapet Åland   

          area  
0   9109841425  
1   7380547465  
2  94206757384  
3   5785648660  
4   1451696825  
Game dataset
   id                  tapahtumaAika  vuosi  kuukausi           x  \
0   1  2017-12-10T16:55:00.000+02:00   2017        12  278773.659   
1   2  2017-09-03T13:00:00.000+03:00   2017         9  397501.008   
2   3  2017-09-18T07:00:00.000+03:00   2017         9  418353.841   
3   4  2017-12-01T17:00:00.000+02:00   2017        12  488970.827   
4   5  2017-09-18T07:00:00.000+03:00   2017         9  418766.580   

             y  

So far we have only done pretty standard file reading.

# Cleaning
We should prune out the least occuring animals, since there is not enough data to teach on.

In [6]:
print("Species on the dataset")
print(accident['riistalajiNimi'].unique())

top_3_animals = accident['riistalajiNimi'].value_counts().nlargest(3).index

accident = accident[accident['riistalajiNimi'].isin(top_3_animals)]

print("Three most occuring species of the dataset")
print(accident['riistalajiNimi'].unique())

Species on the dataset
<StringArray>
['Metsäkauris', 'Hirvi', 'Valkohäntäpeura']
Length: 3, dtype: str
Three most occuring species of the dataset
<StringArray>
['Metsäkauris', 'Hirvi', 'Valkohäntäpeura']
Length: 3, dtype: str


# Merging

Next up is merging the area column from region dataset to the accident one, 
but first we have to edit the columns that we are going to use into uniform naming.

In [8]:

accident = accident.rename(columns={'maakuntaNimi': 'region'})
region = region.rename(columns={'namefin': 'region'})
df = pd.merge(accident, region[['region', 'area']], on='region', how='left')

df.head()

,id,tapahtumaAika,vuosi,kuukausi,x,y,kunta,kuntaNimi,maakunta,region,tielaji,tielajis,tieNumero,tieYllapito,tieYllapitoNimi,riistalaji,riistalajiNimi,area
0,1,2017-12-10T16:55:00.000+02:00,2017,12,278773.659,6980877.550,743,Seinäjoki,14,Etelä-Pohjanmaa,3,Valtatie,18.0,1.0,Valtio,47507,Metsäkauris,13864807779
1,2,2017-09-03T13:00:00.000+03:00,2017,9,397501.008,7503083.730,261,Kittilä,19,Lappi,4,Kantatie,80.0,1.0,Valtio,47503,Hirvi,94206757384
2,3,2017-09-18T07:00:00.000+03:00,2017,9,418353.841,7514104.657,261,Kittilä,19,Lappi,6,Muu maantie,9552.0,1.0,Valtio,47503,Hirvi,94206757384
3,4,2017-12-01T17:00:00.000+02:00,2017,12,488970.827,6722504.397,285,Kotka,8,Kymenlaakso,5,Seututie,357.0,1.0,Valtio,47629,Valkohäntäpeura,4622479273
4,5,2017-09-18T07:00:00.000+03:00,2017,9,418766.580,7506930.833,261,Kittilä,19,Lappi,6,Muu maantie,9552.0,1.0,Valtio,47503,Hirvi,94206757384


Regions finnish name is the common ground on these two datasets,
so we normalized that into uniform name and merged by it.

# Feature creation
After we pruned and topped up the dataset, 
we should add features that are helpful with teaching the model.